# S-CoT Extension — Structured Chain-of-Thought on New Model/Dataset Pairs
## VLM Medical VQA Benchmark — Notebook 09

**Purpose:** Test whether the S-CoT performance degradation observed on MedGemma/SLAKE  
is architecture-agnostic or model/dataset-specific.

| Run | Set `RUN_CONFIG` to | Model | Dataset | Expected time |
|---|---|---|---|---|
| A | `huatuo_slake` | HuatuoGPT-7B (Qwen2.5VL) | SLAKE | ~50 min (T4×2) |
| B | `medgemma_vqarad` | MedGemma-4B | VQA-RAD | ~20 min (T4×1) |

**Instructions:**
1. Set `RUN_CONFIG` in Cell 1 to the run you want.
2. Add your Hugging Face token to Kaggle Secrets as `HF_TOKEN`.
3. Run all cells top-to-bottom.
4. Output JSONL is saved to `/kaggle/working/`. Download and place in  
   `outputs/_archive/scot_experiment/` before running local analysis.


## Cell 1 — Configuration Toggle
Set `RUN_CONFIG` to `"huatuo_slake"` or `"medgemma_vqarad"`.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  SET THIS BEFORE RUNNING
# ══════════════════════════════════════════════════════════════════
RUN_CONFIG = "medgemma_vqarad"   # changed to "medgemma_vqarad" for Run B
# ══════════════════════════════════════════════════════════════════

CONFIGS = {
    "huatuo_slake": {
        "model_id":      "FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL",
        "dataset_id":    "BoKelvin/SLAKE",  # <-- REVERTED TO CORRECT ID
        "dataset_name":  "slake_scot",
        "use_4bit":      True,
        "max_new_tokens": 200,
    },
    "medgemma_vqarad": {
        "model_id":      "google/medgemma-4b-it",
        "dataset_id":    "flaviagiammarino/vqa-rad",
        "dataset_name":  "vqa_rad_scot",
        "use_4bit":      True,   # NF4: ~2.5 GB, avoids OOM
        "max_new_tokens": 200,
    },
}

cfg = CONFIGS[RUN_CONFIG]
MODEL_ID       = cfg["model_id"]
DATASET_ID     = cfg["dataset_id"]
DATASET_NAME   = cfg["dataset_name"]
USE_4BIT       = cfg["use_4bit"]
MAX_NEW_TOKENS = cfg["max_new_tokens"]
OUTPUT_DIR     = "/kaggle/working"
SAFE_MODEL     = MODEL_ID.replace("/", "_")
OUT_PATH       = f"{OUTPUT_DIR}/{SAFE_MODEL}__{DATASET_NAME}.jsonl"

print(f"Run config : {RUN_CONFIG}")
print(f"Model      : {MODEL_ID}")
print(f"Dataset    : {DATASET_ID}  ({DATASET_NAME})")
print(f"4-bit NF4  : {USE_4BIT}")
print(f"Output     : {OUT_PATH}")

## Cell 2 — GPU Check + Install

In [ ]:
!nvidia-smi
# Do NOT pin transformers — use Kaggle's built-in version which worked for the v2 baseline
!pip install -q accelerate bitsandbytes datasets sacrebleu tqdm
import transformers; print('transformers version:', transformers.__version__)


## Cell 3 — Imports & Device

In [ ]:
import os, json, re, time
import torch
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')
if device == 'cuda':
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name}  {props.total_memory // 1024**2} MB')


## Cell 4 — Hugging Face Authentication
Required for `google/medgemma-4b-it` (gated model). Add `HF_TOKEN` to Kaggle Secrets.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('HF login successful.')
except Exception as e:
    print(f'HF login failed (may be fine for non-gated models): {e}')


## Cell 5 — Load Dataset

In [ ]:
if RUN_CONFIG == 'huatuo_slake':
    ds_raw  = load_dataset(DATASET_ID)
    samples = list(ds_raw['test'].filter(lambda x: x['q_lang'] == 'en'))
    print(f'SLAKE EN test: {len(samples)} questions')

    # ── Diagnose schema ─────────────────────────────────────────────────
    s0 = samples[0]
    print('Dataset keys:', list(s0.keys()))
    for k, v in s0.items():
        print(f'  {k}: {type(v).__name__} = {repr(str(v)[:80])}')

    # BoKelvin/SLAKE stores images as img_name (e.g. 'xmlab0/source.jpg').
    # The actual images must live at SLAKE_IMGS_DIR on Kaggle.
    # Attach your SLAKE images Kaggle dataset and set this path:
    SLAKE_IMGS_DIR = '/kaggle/input/datasets/shriyanshraj/slake-imps/imgs'  # <-- adjust if needed

    # Check if images are embedded (PIL) or path-based
    _img_val = s0.get('image') or s0.get('img')
    if _img_val is not None and hasattr(_img_val, 'convert'):
        # Images ARE embedded as PIL — use directly
        _img_key = 'image' if 'image' in s0 else 'img'
        print(f'Images embedded under key: {_img_key!r}')
        def get_image(sample): return sample[_img_key].convert('RGB')
    else:
        # Images are path-based — load from SLAKE_IMGS_DIR using img_name
        _name_key = 'img_name' if 'img_name' in s0 else 'image'
        print(f'Images are path-based. Loading from {SLAKE_IMGS_DIR} using key: {_name_key!r}')
        if not os.path.exists(SLAKE_IMGS_DIR):
            raise FileNotFoundError(
                f'SLAKE image directory not found: {SLAKE_IMGS_DIR}\n'
                f'Attach your SLAKE images as a Kaggle dataset.\n'
                f'Available /kaggle/input/ datasets: {os.listdir("/kaggle/input/")}'
            )
        from PIL import Image as PILImage
        def get_image(sample):
            return PILImage.open(
                os.path.join(SLAKE_IMGS_DIR, sample[_name_key])
            ).convert('RGB')

    def get_question(sample):  return sample['question']
    def get_answer(sample):    return str(sample['answer'])
    def get_is_closed(sample): return sample['answer_type'] == 'CLOSED'

elif RUN_CONFIG == 'medgemma_vqarad':
    ds_raw  = load_dataset(DATASET_ID)
    samples = list(ds_raw['test'])
    print(f'VQA-RAD test: {len(samples)} questions')

    # VQA-RAD from flaviagiammarino/vqa-rad HAS embedded PIL images under 'image'
    def get_image(sample):     return sample['image'].convert('RGB')
    def get_question(sample):  return sample['question']
    def get_answer(sample):    return str(sample['answer'])
    def get_is_closed(sample):
        qtype = sample.get('answer_type', '')
        return str(qtype).upper() in ('CLOSED', 'YES/NO', 'BINARY')

closed_count = sum(1 for s in samples if get_is_closed(s))
print(f'Closed: {closed_count}  Open: {len(samples) - closed_count}')
print('Dataset loaded OK.')


## Cell 6 — S-CoT Prompt Builders

**Identical to the original MedGemma/SLAKE S-CoT experiment.**  
The 4-step structure is word-for-word the same to ensure comparability across runs.


In [ ]:
SCOT_TEMPLATE = (
    "{question}\n\n"
    "Please reason step by step using this exact structure:\n\n"
    "Step 1 - Modality: Identify the imaging modality (e.g., CT, MRI, X-ray).\n"
    "Step 2 - Anatomy: Name the primary organ or anatomical structure visible.\n"
    "Step 3 - Observation: Write exactly one short sentence answering the core "
    "question based on Steps 1 and 2.\n"
    "Step 4 - Conclusion: State your definitive answer (if possible, a single word) "
    "in the exact format 'Final Answer: X'."
)

def build_scot_prompt(question: str, is_closed: bool) -> str:
    prefix = 'Answer the question with yes or no.\n\n' if is_closed else ''
    return prefix + SCOT_TEMPLATE.format(question=question)

def extract_final_answer(text: str) -> str:
    import re
    match = re.search(r'[Ff]inal\s+[Aa]nswer\s*:\s*(.+)', text, re.DOTALL)
    if match:
        answer = match.group(1).strip()
        answer = re.sub(r'[\*"\' ]+', ' ', answer).strip()
        return answer.split('\n')[0].strip()
    # Fallback: last non-empty line
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    return lines[-1] if lines else text.strip()

# Sanity-check on first sample
s0 = samples[0]
print('=== Open-ended S-CoT prompt ===')
print(build_scot_prompt(get_question(s0), is_closed=False))
print()
print('=== Closed-ended S-CoT prompt ===')
print(build_scot_prompt(get_question(s0), is_closed=True))


## Cell 7 — Load Model
4-bit NF4 quantisation applied for 7B models. MedGemma-4B loads in fp16.

In [ ]:
!rm -rf ~/.cache/huggingface/hub/*

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# MedGemma-4B NF4 = ~2.5 GB. Single GPU, no cross-device issues.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
print('Loading MedGemma-4B in 4-bit NF4 on cuda:0 ...')
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='cuda:0',
    trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()
device = 'cuda:0'

used  = torch.cuda.memory_allocated(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'Model loaded: {MODEL_ID}')
print(f'VRAM used   : {used:.1f} GB / {total:.1f} GB  ({total-used:.1f} GB free)')


## Cell 8 — Inference Loop with Checkpoint/Resume

In [ ]:
import os, json
import torch

torch.cuda.empty_cache()
print(f'VRAM before loop: {torch.cuda.memory_allocated(0)/1024**3:.1f} GB used')

# Wipe stale output file
completed = {}
if os.path.exists(OUT_PATH):
    valid = []
    with open(OUT_PATH) as f:
        for line in f:
            try:
                r = json.loads(line)
                if 'error' not in r and r.get('prediction','').strip():
                    valid.append(r); completed[r['idx']] = r
            except: pass
    if not valid:
        print('Wiping stale file — starting fresh.')
        os.remove(OUT_PATH); completed = {}
    else:
        print(f'Resuming: {len(completed)} done.')

errors = 0
f_out  = open(OUT_PATH, 'a')
eos_id = processor.tokenizer.eos_token_id
pad_id = processor.tokenizer.pad_token_id or eos_id

def run_inference(image, prompt_text):
    """
    Correct MedGemma inference:
    Pass image + text directly to processor (no apply_chat_template).
    MedGemma (PaliGemma-based) handles the chat format internally when
    you pass messages=[{'role':'user','content':[image,text]}].
    """
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text',  'text': prompt_text},
    ]}]
    # apply_chat_template tokenize=False → get the formatted string
    formatted = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # Pass the formatted string AND the image to processor
    inputs = processor(
        text=formatted,
        images=image,
        return_tensors='pt',
        padding=True,
    ).to(device)

    input_len = inputs['input_ids'].shape[-1]

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=pad_id,
        )

    new_ids = out[0][input_len:]
    # Decode: try processor first, then tokenizer
    raw = processor.decode(new_ids, skip_special_tokens=True).strip()
    if not raw:
        raw = processor.tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    if not raw:
        # Last resort: batch_decode
        raw = processor.batch_decode(out[:, input_len:], skip_special_tokens=True)[0].strip()
    return raw, new_ids.tolist()

for i, sample in enumerate(tqdm(samples, desc=RUN_CONFIG)):
    if i in completed: continue

    try:
        image     = get_image(sample)
        question  = get_question(sample)
        answer    = get_answer(sample)
        is_closed = get_is_closed(sample)

        raw, tok_ids = run_inference(image, build_scot_prompt(question, is_closed))

        if i < 3:
            print(f'  [idx={i}] first_toks={tok_ids[:8]} raw={repr(raw[:100])}')

        prediction = extract_final_answer(raw)
        record = {
            'idx': i, 'question': question, 'ground_truth': answer,
            'prediction': prediction, 'raw_output': raw,
            'is_closed': is_closed, 'model': MODEL_ID, 'dataset': DATASET_NAME,
        }
    except Exception as e:
        errors += 1
        print(f'  Error idx={i}: {e}')
        record = {
            'idx': i, 'question': '', 'ground_truth': '',
            'prediction': '', 'raw_output': '',
            'is_closed': False, 'model': MODEL_ID, 'dataset': DATASET_NAME, 'error': str(e),
        }

    f_out.write(json.dumps(record) + '\n')
    f_out.flush()

f_out.close()
valid_done = sum(1 for r in [json.loads(l) for l in open(OUT_PATH)]
                 if 'error' not in r and r.get('prediction','').strip())
print(f'\nDone. {len(samples)} records | {errors} errors | {valid_done} valid')
print(f'Output: {OUT_PATH}')


## Cell 9 — Quick Metrics Preview
Sanity-check before downloading.

In [ ]:
from collections import Counter

def norm(t):
    import re
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', ' ', str(t).lower())).strip()

def token_f1(pred, gt):
    p, g = norm(pred).split(), norm(gt).split()
    if not p or not g: return 0.0
    pc, gc = Counter(p), Counter(g)
    common = sum((pc & gc).values())
    if not common: return 0.0
    pr = common / len(p); rc = common / len(g)
    return 2 * pr * rc / (pr + rc)

def closed_acc(recs):
    if not recs: return 0.0
    correct = sum(1 for r in recs if
                  norm(r['prediction']) == norm(r['ground_truth']) or
                  ('yes' in norm(r['prediction']) and 'yes' in norm(r['ground_truth'])) or
                  ('no'  in norm(r['prediction']) and 'no'  in norm(r['ground_truth'])))
    return correct / len(recs)

all_lines = [json.loads(l) for l in open(OUT_PATH)]
records   = [r for r in all_lines if 'error' not in r and r.get('prediction', '') != '']
err_count = len(all_lines) - len(records)
closed_r  = [r for r in records if r.get('is_closed')]
open_r    = [r for r in records if not r.get('is_closed')]

print(f'Total lines   : {len(all_lines)}')
print(f'Valid records : {len(records)}')
print(f'Error records : {err_count}')

if not records:
    print('\nNo valid records — inference may have failed. Check Cell 8 errors.')
else:
    f1_all  = [token_f1(r['prediction'], r['ground_truth']) for r in records]
    f1_open = [token_f1(r['prediction'], r['ground_truth']) for r in open_r]
    print(f'Overall F1    : {sum(f1_all)/len(f1_all)*100:.2f}%')
    if closed_r:
        print(f'Closed Acc    : {closed_acc(closed_r)*100:.2f}%  (N={len(closed_r)})')
    if open_r:
        print(f'Open F1       : {sum(f1_open)/len(f1_open)*100:.2f}%  (N={len(open_r)})')

    print('\nSample predictions:')
    for r in records[:5]:
        print(f"  GT: {r['ground_truth']:<20}  Pred: {r['prediction'][:60]}")

print(f'\nOutput: {OUT_PATH}')


## Cell 10 — Download & Next Steps

1. Go to **Output** tab in Kaggle → download the `.jsonl` file.
2. Place it in:
   - **Run A** → `outputs/_archive/scot_experiment/FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__slake_scot.jsonl`
   - **Run B** → `outputs/_archive/scot_experiment/google_medgemma-4b-it__vqa_rad_scot.jsonl`
3. Tell the agent — it will run `scripts/scot_extension_analysis.py` automatically.
